# 07 · Structure-based ML & Glycan Embedding (Task 02 — Phase 5)

This notebook implements three analyses that bring the project into direct alignment with the AI-for-Chemistry course themes. All steps operate on **molecular structure** (glycan IUPAC sequences) rather than LC-MS intensities, demonstrating that the course's methods generalise beyond small molecules.

---

## Workflow

### Section 1 · Structure-based Disease Classifier *(Course: Weeks 1 & 2 — molecular representations + supervised ML)*

**Goal:** Train a classifier that predicts cancer association from glycan molecular structure alone, using the ~50,500-entry `df_glycan` reference database as training data.

| Step | Detail |
|------|--------|
| Feature extraction | `annotate_dataset(feature_set=['known','exhaustive'])` → 175+ dim structural fingerprints for all glycans in `df_glycan`; this is the glycan equivalent of ECFP/MACCS molecular fingerprints |
| Label construction | Binary: `cancer_associated = 1` if any cancer in `disease_association`, else `0` (≈ 602 positive, ≈ 49k negative) |
| Sampling | All positives + stratified sample of negatives (target ≈ 5,000 total) to keep compute manageable |
| Models | Logistic Regression (linear baseline), Random Forest, XGBoost (primary) |
| Validation | Stratified 5-fold CV; metrics: ROC-AUC + Precision-Recall AUC (primary for imbalanced data) |
| Interpretability | SHAP values on XGBoost — which structural motifs drive cancer predictions? |
| Application | Apply trained classifier to the 5 discovered biomarker glycans → predicted cancer-association probability |

**Input:** `files/data/glycan_embedding/df_glycan.pkl`, `files/data/glycan_embedding/glycan_list.csv`

---

### Section 2 · Structural Embedding Space & UMAP *(Course: Week 4 — unsupervised ML for chemical space exploration)*

**Goal:** Visualise the structural space of glycans using UMAP on structural fingerprints, and locate the 5 biomarker glycans within that space.

| Step | Detail |
|------|--------|
| Dataset | All disease-associated glycans from `df_glycan` + random sample of unlabelled entries (≈ 2,000–5,000 total) + the 5 biomarkers |
| Features | 175+ dim structural fingerprints from `annotate_dataset` |
| UMAP | `n_neighbors=15`, `min_dist=0.1`, `metric='jaccard'` (appropriate for binary fingerprints) |
| Colourings | (a) cancer / other disease / unlabelled, (b) glycan type (N/O/lipid), (c) sialic acid count, (d) fucosylation status |
| N-glycan benchmark | Load `N_glycans_df.pkl`; overlay in UMAP; verify N-glycans cluster together (structural coherence check) |
| Nearest-neighbour analysis | For each of the 5 biomarkers, find the 10 nearest neighbours in fingerprint space; report their disease associations and tissue labels |

**Input:** `files/data/glycan_embedding/df_glycan.pkl`, `files/data/glycan_embedding/glycan_list.csv`, `files/data/glycan_embedding/N_glycans_df.pkl`

---

### Section 3 · Biological Interpretation

Connect the classifier predictions (Section 1) with the embedding-space neighbourhood (Section 2):
- Which structural motifs are most predictive of cancer association (SHAP from Section 1)?
- Do the biomarker glycans sit near other cancer-associated glycans in the UMAP (Section 2)?
- Are the motifs that drive cancer predictions also the ones that distinguish the biomarkers' neighbourhoods?

---

## Outputs

| File | Contents |
|------|----------|
| `files/results/embedding_results.pkl` | UMAP coordinates, nearest-neighbour table, classifier predictions for all 5 biomarkers |
| `figures/fig_structure_classifier.png` | ROC + PR curves; XGBoost confusion matrix (Section 1) |
| `figures/fig_shap_structure.png` | SHAP beeswarm of XGBoost structure classifier (Section 1) |
| `figures/fig_structural_umap.png` | UMAP coloured by disease, type, sialic acid, fucosylation (Section 2) |
| `figures/fig_nn_analysis.png` | Nearest-neighbour heatmap for the 5 biomarkers (Section 2) |

**Input:** `files/results/enrichment_results.pkl` (from NB06), `files/data/glycan_embedding/` (reference databases)